# NimbusPay: final reproduction
Final adapter: raw-data 16-bit LoRA; dev EM 71.50%, field accuracy 94.56%.

Outputs below are from the candidate's completed Colab reproduction (62.26 minutes). Packaging subsequently changed only workspace/bootstrap and final summary cells so reviewers can run without the candidate's Drive. Their outputs are cleared. Training/inference source is unchanged. The original executed notebook is preserved byte-for-byte in evidence/notebook_executed_original.ipynb. The portable bootstrap has been syntax-checked but not rerun on a GPU after packaging.

Select a T4 GPU, run all cells, and upload only the original training_assessment.zip when prompted. Source and selected config are embedded below; the training code also remains readable in this repository.

In [ ]:
import time, sys, os, json, subprocess, shutil, zipfile, io
from pathlib import Path
STARTED=time.perf_counter()
from google.colab import files
ROOT=Path("/content") / ("nimbuspay_reproduce_"+str(time.time_ns()))
ROOT.mkdir(parents=True)
PROJECT=ROOT/"project"
PACK=ROOT/"candidate_pack"
print("Fresh workspace:",ROOT)


In [ ]:
import base64
SOURCE_ZIP = "UEsDBBQAAAAIAE+7bF0d67jhhRQAAE82AAALAAAAcGlwZWxpbmUucHmtW21z27aW/p5fwZs7O6ASmJadpNNI5Ye8tc1u02abdO7uaDUcmIQkVhTJEqRt1eP/vs85AN8kOc3du+7UJkHgADivzzlAHj9+/KbI1JX3+blXVyrN03ztqTzxKnXjrSutk72X5itd6TzWcy8vvKKpy6b2Kl2qtAoeP378KN2VRVV7qlqXqjJaenFR7vHbXEtvHUtvo8wmS6+k97spcuntVL2RXmGkZzZNnWb421yVVRFrg7Y63elHq6rYeSX6YZjnyH/EaztV3uzKvaeMl5dtU11U8cYO3DTrNbaxUrGONk1HYFckOouwmcJ2w3Zzsyqqna5M28d/1dTFB+r4fVG9UY1R2U8fpEetn4utztM/dSW912ltXuXJ632tzZsiX6Vr+ch78OczsZWGfXb8fVWtm53Oa+KAriMDJkvvB40+qk6L3FKcOCboVd0u7qeiUm467yPaeZ1gMWhQt2hn30sSTaXta4QNRturtI5a6Vq6Ji4q3RLOCoUVcBOGWxkatcPv/3j3358effjl7bufQvGfNzo/p1+XwYuzafDi9dn73NRVE9fi0etXn96FSRrXPs8a8hD5R6Py+nn4vcpAEdzehhffSJWVGxU+u5RZFV7qs+dSl0W8MeGlJEaEzy9lnGmVh5+rRsu42JUqri2JyaN3//Xx3a/vP7z7+fOn8O6RJ7hntCovvhEznp3WMYEwBNT3sN3RtaRkN/qPDGwddnOrpvkdqXz77PKIGu/n2aWlZJd5PKNbvqN1/+hRoldeAu31S3k7mbHWkGL75SQgoeV1sNsmaeXbF2O5oG9TU0fF1tKZ17sy5EHgPgY+FQFaBLcHN1Va66jWt7VP1hbQXMa/lWmegB6YrHPTQDmUidPU8cKOhNZksBgQ5EXS6AyrrIobM5m1axyQFyL4vUjz4TTVKepPxf/kwoMaehU8icf07BR1a1F+vFo7ZtThyNYCUtYIGs3aqxPquRCsY2IpK32dGlhMyK3tm1hiQ+BmEvEEYR3owtjnOQg1Ve7VVhBXyjgzGaxge2MV+SHikj1NlNT7Uof8HKxgQDVUO0En+JydKsM7IWbTe6nqOo9gZZkmg2fzDoVJSiUmPFm68pi81TmxnG1v3Ev6J/eOYjZ4sQyPnY5PdguHFj2HfVtNucqv+C1iGhGvUeSr56L/0mDPSdFcZdp2OhhIKttAxif2Z5fsWHjKUX5ZWk+ebG8mlvEIJmiMoAs+/pf1gZ1bQezM2oQUSoJE65IeqDPoIU6otTaQc8dDO3xGQxZnF8sFLDKvwXHwbaCf/EhMM/6JnpMT2gufBEtUYIMJfSHx30xM7LSIWLuyhnapssz2UbxR4LeGqFWtLfnZJSmL1WTLZZUk0bpz9JEjwVbNNFdNln2B4ldRc66SyCljNNw7UV3MMp37tstkGYb2SYo3mOSsncS7Kpo8UdXe26UGYTreCKYDfKAzE4Jf0+nyyYDQUyY9aJgthxOrfO9f/y2kYewArskBWGIjXWJ7S3PoXZQmJiSiZDmQCu1rp8w2XFzYienbRLoFtaQexRlm9ABkMhKVVR/SswjWAZ2OfKOzlcTy6G8AoQ16xCrL2h76VpGtGqeB9JOHhFeCWKeZv1O3Pi1CL0S3WugN703T3rrx599Onnzb0XDbvNvOrD1hZ6ao/IVebJdPFz7zB1q8DUNhtyQ8/NKeXW3vyjBf26+f33adTpZP/PzMrm67nJxYlFvoltqGG5BizGs0uFUsXcCCRfexHdZKPlwSPJNVA8Lhz0XeGi3w4C95tveKKgUGU5ln9gbqdQ6vU3mt5Uovh8pea69VPNlCzV6dGVmyG6BJA32tMn8yty/WJwbkyWIVb6w5cGeMDw+RlJ8UkWEWOJsGeoyutNqZ8EJCpFGubyx/AUKm0y9AuS6OgG/DsIJXORRSOJZZq+umyepwsWydlo2p+DUJOMAbf6B2JKkMPpSENehYaaJL4XcSmDJLa+ozGshBtNrPPATqXTjweNRzMuqmb2Nd1hzpg3//9MvPbzV55XdVBRPyrjDVdtTdGTWpmN3L5Dt+poDOKQPNCMVK4HRDal0Mui7tB+mJj1aZIB7v/Vvjwcvla52IU1Mh4gJhKmQePtEOgHR9YfMPMZFAP+P92KnIe+o84RH9dwdnWGftipwTT/PaX4m7AY9zIN/7mXc3WP39+V231XtA42udtEaBfcCEVlljNgNHfpPWG68oyTHSlALxntKV1VjAKQMi2r8/mEx2Ux2ItbJcTZej1lNhchgkKQxNTnF3AWFVRabFkhezo8UQMUhvIazZwhWQ4YrxjNBdtnvIBwlh7ZH529A7HcbTpyGQH7DfU+48pkDOx/yrkY4/Wd8aWZdqQlHWom0j8ViZBHXhi7hJWtjV/rCYrE/uklwGhLAo6Flo3Y2bWftPntiFy8FarC8K0XKgjOoG+0vYpHzQWkylHTyKHYHZqFIjuM2W0mzTMjKljlOVtf7IJkGcpzRl5JhiJzZAPPC9NtwHMIa09MdLYA9wR3Y3q5z5tdYzw/Lu5ycMZr6yGH8I6+nDg8ieRrD2H06+8vz06cXk3y6mYTidtYbmnDPs5w5fR3Z1bEUubtpl2lDEyDKCwvlkphIAE5433krQv6ZAgv05q2lTawtC6YkSg0SbOkSncxreemKfWs9hM2BibALaujjllEvKAorGDN3q6bEDP92zxdldS4VNxWJ7yjGAxfi19YhzD6bnKW+F7W/aqsuqyBLY42BFxFbxm6GyDaHgTBNvqyafCckcGvC0zX7aBTgQZL4i5zzglAvADzLK7fSYTeOBQy5ZHnjik65rbKYLDDPHBgRp2taIA5xIHxMmrRi6d/Hp86tfP7//+YcTPHFJ5zgVbXdLzs3Rb6snUcvkhzdP2hzSrn2nlG50xHbM4zLRZX7cCBfMKI5U+VwkqlbnPMR17lUI6HgxTJ4odWIqNgFqwWiXaS/nsK91vcEwNrQTwJX7JqZ38YBE6a7ZHeNeojM5f3Y5efLs8lDUbtB34bcXLy+leJ+TI6PKEvTS+r057HjVsKKaFNRq2HKTx+zMeoUeiNTN6GR6J2g/Ytb5CylKXcWEWxFdXNHlT/jAhXgxRdx6+YJ+vcQvoGt4PiTlPuexMi+DfmS7L7l4MZUvX8iXLyUlORP8SEHg0Og/IttHzNwmpXAr10lkFzW970VkwwkFmwARrd5biArsOvgAk+aindpGO70rqn0EmENaZAFueFCYOGS2aXZ+GQDG4rsVYsnBm6MV5avwRroCucl34fNoOp22/3eEjuoOdt6/KB3aBKBfDunszOHzXFFFwWpXpf9oUpBZV2T6ff8H0DtHk3GncFzUdJlHXwD1K1eWUfkWwqUSXmQLi9zMj217UhUlBfNpMH0hr1IFkJAjgAhZI9lxJZI3r3779Oqn6KcP1FrR5JiRcxvwiPDG71Ckbftw3T4U7cMaytA+I1C7p6S4ye3zchAGcmZn+NVCJGFReY55apipk7nTBEvru/DFsYixDXLlByVn3wYSYKMqpPKh854bHW/LAs7SAFhTfmQ9livNMk/tM5gK44lcpcv2uqJKQWQIrV1IWl6KmSIVx82uyRxYqXVpqPo7wghwfRU7VUIEdpasIrFVkcGSSACVE1BcwHNAZjeq2oHBDL5CEuiNTtcb7EfHao+GC0k1WBvCRnN16+r3ipltx9Pfou0N8/COMDB0WhOzAWxnrK/3sijrdBcKlagd5Y8wbnGwvYLPIdrNTyUlDnij3a73oeB20bZSn8sX9q0uamDALN2lVLKFURYVrLBoFZcL5QNYIyliRCdbKQ6BiSTSm6LaQqHC6cBX2aOJ0B1RuOK9NTbeO/2SVspEDV4rTIydLnbVlrAtu/j1QMmPTlgCN1dERz7deQtZWqbMkPODqglMRIcnuvgPaO5kCFGGH7oobcMsJSw9D8BlEhIyJ+iZnTQBM9dkL0Ca2Z6Dtf1gxwuqQnVe8QiNtUbn9WuYCekIwM6FhXSIIceQ9yCCmH0eb6qCoAmCA4JEBayS7jRFsBVE0CDXqii2WOZaJvt2qoirsf0aQts8CkRj+mBEU8Pn+SemOOPJJ+ff9B6G4lc4IEbh0oUzlWUFB0h/cn755MmzKeUaukLe/MCA9nPb/yBmsFUMCstWxjC9EosjE1rBhHSFzMmlR+1RyV+MnJwCHkdYcj74aI0Bpu2+t5ynAK4BdtfRBqqGLZ0k/QCMlLa85tjv/krmL/2SHfPaB+mCiP0je8hmAcpIp4cvAwv9u/erJu/g1RvtShoQcLoCjKSSYVxnfMJKhbeGWYpsOcu8pJDelV7R6SEdGIJy0G9UZ61LkSy3+ToOyE8ACo4B0AgZWbidr4oTedWDuPsowxpyTR7wqWOQT9MststhJdQNFAQoFVU/28F4tMMFQ0Ei0J45jBjMNMWwyRVMnD9451hIyNfy2WrfQ+mI9cLdCe/RqcoBOpQHKj2nylR4WLO1CUmfXyCCt9mFIzAoaEWDrzaZprNhE/If/0ukiMjIXuxIp+b2xe3SZswIr+QIZ8yJNiOecT7MWhjZc4iZHUrQLhPLxejbEuLhc+BUZwnhjqPeB5+XVtTRumyiH9LXYsY6Y9ta4dsPnR71itiqy6xVuLHkZ8M3N4yhcQ/psFmrlQ8lGPeH2e2ouiDdGwma9GtQqXFfutPeyZFukYn+E6bZlgwc4f7Y1Nadu/JLX3bpCiyi6ybm/59Fhv9rbeHrKgaYHjrjSK7EXX4fDTV4gCTIg+TWg6zSW/YWXOiEi5gdIoJfdXNQoemZc6JC868ycn5Y0DidVs7XRZaEX7BmeyZJ+yTjdDVfbNhvd8wHP9J3G5eOzKc3P7778CrYJQdiWI4qaccuipYjO8ZTFd4txE49dCtth+jYvfhMhv3Q5C9lPv8n7cEagMpVtjepgUvBuvoJB2ZwtQ/vXNGViq+u7LrsKy7U/36+UmnWHgltCKvCHRJuWgNAiFlXRiG90RxAUi6uGz5vWVWq4XOIWCdUuQ4ohFUpgnJNg98Q8kN0J7BJK0S6tVYZN6iaCbQzHdBQOwJ9cFip0R2dJDV0LwDBi79KaoAXi+lo3F4Ya0qtzVldnPFArDm/hrOjozzQrG85g+jp0YwpBcOEz/8qndkjQXo1kg6g2WL2SBGZ/B9NQbPrHThmWopU2nb0fv7o5cg0OgDIg5qc6VJFVLvyPo+l2mKus26wvi2zNE7tHTGPlt0Ye+0NUuLGgE9F83Wj1v0efsQKsxRQnuaiFZc6y6h4Sb1zJGEm2jQ7lXcDPl4Q/u9mQ2JHoqATEzAyEPedwbGKkF4Naqpx3SjY8J6PwbqCPsX7kO9s+bYHzHqIo8ZnQQc3HuY3wP7rcLHt8RDd+OKKA53ukH17/IVKoaS19EbPdD3ML4Gk5JqOm/vqIYYyUc8WNPoTImYSEki/nPwtpL8008EJF5lDexxxcHBRp/FW19ww2M9oO1IQx8QM0IGuf7VHHZYtUvCyLApACOY3KZJUrfPCgLo7VcKnBVviCCVyb0wQYxMpqWhEQlS5rV7OxOdf3v4yQ0drrjVSAI9PclZpzJtqKj13N0ZgLcDOxBUFA6SLM96AFpTAei17YMn+3xEwrTek94Fr4z6aTm0j5Hc7Ve0fdIfsvXaUIjrGqzJsr20GbY3oI71RUqnKgA7elGv3xdkZmQI856ZIY66N0coxCYNmbC9pYs33srhWlbhodIIOBQrxNf1sBf1ruoINX9PtCoHgzMFMBJuVokN5Qa3RCECcGgugagaDpLsBN7iVOMG4ECOZozS2LX861DKIKwgf6hqSJHDoT+gmTlHAbXa3cX/4+Ju3SitTiyEB8fm5IIUcEKICiavHUUz0pyD2Ct5zndO6vbZwCJqWkh16BQnAzEybide7BqaHtP0mqlfPLm1Rdn7cN8+Petlc5+AULqSCO52qy4UK+CP+uM8qQAd4H/z+SlS4AYrg0MZVrJAuFtHgdVZc+eLJ+UhyLmOhcgWVG05US5zJ5NcpzJqYFNS3tRjdbuyvJAdcQXHexF+IMi0h+VWl9Z+k69TbLpSOQyG0mSeeflE6PdAMyJzCsLOiwRmSu27ICMOiQFuTbz/YvR71H3n+IY0REBteZ7S1rWHPca2svzXt86XeCR1Zj0sbo+F0UNMSn7VPg+MRoK5wcIm2XXf7MJk/kF30BSGXNy4GR8G2fjKww0W+PEH7+Ki4R/EqIOu2l2ropt8gpJFzoNJKl15t9T7M1O4qUd5u5u8Os9HdccIJvwCnaXTOsiSCC057l3Pih31vT4MtzFXBwCfJ0SkQ31pwOnwL1U65sh/E5ho6zJdOxI2Qub4hNobi6OoJR1oOg6FdxTjVPsqmD7LlE7nwQQ78QNq7nN+EWGTwFsL6B5la5a8kT0OWYUK7Jpg/RhsV28uycGNkGoAq1jw3mgrafvdOB3CtYCyL+FRqpG8Do+B7PiPejnOUea9VSHKA88XpOwZDcbqOPOXfvdcFpENWYyiu8z+MgIEhyrsCjbsqZTzmNmFc+qcWg+KLrafBATEGLtwc/buY98/OfR6kf/bfVQR0F6iGn/LtWof10o7AoBGETHTC97I6DkccBUubSj2QdH59HevUqly5jSzwMFGlRnvg2CKjB0pf1HFEfFjpoo+jfHdwzY1RUxjSI3Wzt9wWpYWk/ZEddaNLU9XgAwN4GrR0avEJKs7wKetgP6cEaq3okpvHvpNNG+Lmf4SRWhCNVJufOP5Y3eCRJ7e5mL1YStaXiou7tvzfscftzY5nAjTAbnuc036pxsffWp16oGo44LeLx71O9vci+PID+Q2qp5P/Gonp6OtIs3sqyILT+hD7DnXp+PshfIYXZKDQ3jVgnDuzARrejI/jusLfw8cjUJfzb6ayO0UCXql0YzSSkjGCkaLJ6VyOknEdQeTVX9H97mL67XTKGki5w5jc/ei6y/dwzmajk8D7FZFP33icHCCpZofp2eNEybVooqWvimLr/eP95x+dkpnA+9SxbyZGokzpyjS56ygCcIkiSiaiiCqXlFQ8+l9QSwMEFAAAAAgAT7tsXY+SpwQJDQAATh8AAAgAAABhdWRpdC5weZUZ247jtvV9voKzaUBpR9bObJICtaEdbLNpMW0wu9gsWqS2Y9AWbTPWbUnJHtX2S/+qv9Mv6TmHlCXNeDatAdskRZ77lXrx4sU7WUqdqkyZUi3Yp49v7+4H7+9//JmJKlZlyO7lVmqm0iLXpYzZvGYqW0ots4UMX7x4cWGfMKFXhdBGBmyRJ4lclCrPDE6KGn7NNmCxKGWpUsmEYXEZsLUw60TNA/arybOAaXmx1HnKClHissPIPsDUPojlQqUiaR68s9OLv/7w80/RmC8A+irXNQ94oVWuVYlDkeZVVs4KoYyEafmQzZAMN1QxDBZrkWUygVEislUlVvg0kzI2s3WVioxPL75/++mnaM8LUacSoC2FSiQe1XJZZThYahAWolssCB/8S2NgYVMv4DdfgsBwmpdrqfnxIpZLluQi9gp/CIyXlc7YGMUQ4qrxjM+WuWYGRE0CgH2hliKelfKh9PzQFIkqE5VJ4/lMLZkJTalV4flTAr0D7qVXBDrfGUDQQKBlC4Lz8NdcZR7hjKu0MJ4OZGYqLWfCLJSK/iQSI/0rPsk40aKRFgLoE455Htcewmo5wL0WbEJHEjyCW57Qm+UlS8LEEQ3EC12anQIy+RvuEGgJWtPGqwOL5YLBp47QoMJYygIHXu2PYKMALkw0no5YGbV00QGEVFSltwm2wW5dOzD4ATI2RAgQWTMgt6wL6dXjzRRINPSEVrY+PsT1y2jbHsePwxyKopBZ7C35HlAch2y/ObJ9Ha4k4PUv9ZEN3rD9FgbcHyGgaHvhKCBb5ZYEkcWstd0TaS3KPImjOizywrPH/BFy1jH3PU/yHR/yD9+AraUyVlWKs9cwW6vVGsc3/Eh0AawAvj5YvVyJRc0aKGwj61dbkVSS+w2VlpXWxXwkbP+stVsjb+lGW9jgkfH/44/TodXbfZ5JcMg8K7XIjKDAwlpSCIsEW+1pVhmVgVVBjPIc8SdUfgBm55O0tQyXVZKkolysPc0n8f718dXp99sjD+pxe3Dq97Vf6ppI7HARl2ET5dAjCxx4PRgB/zp+9XX66uufuU97wfyVyUFEQIXXqgMfMbvqeGw+8mEhi5L9DVX0g9a5HhbCmN9mHoRrWe9zkSGORP1TxhGIw1Rzj49/eTv4x/XgD1PQxkkEcHwaVmDo2vP7BKllX5D8/sMYT+9vrkGCLXy/lRYqun2A2nVj1lXy3bsO61+x91lSA/cQShaq7G4cWKuCUJMWCcjNjFhmkxYmKkbJYy60ZFmVziEQhyegdiHS3JvE40kcTF96t8NJOImv/Ft/Yl56idisze1kfthM5v4tPx1cVBozYI1Hb4f/+de/D3f3H+HA4aOZhLcw6G6GjAbRCexrDnvBOXpMOtqVOeATN9sJc7BDADW8PShaQnvAIe7NS5EghfyqIeXKMhP0dNNS6h5fARkGudRVIaU5wJ+opX9rFzeiZCuYHxAb0kOB4KDsxFJ0wKBO6CEzNlhgn4kMmJrf83rgHP0eBdC3OnyYUkYBy1EZFBpgV7AtKANYuXtkoyRESLbqISrHqXjwroPUJg3PH3z7nT88zaaQQHdook/OWys1UmjydW8uEvSQQ6JSVR6WUvrjX8LL20k2ffk7Hlhs/nABYUdllXwCbhu5+sNLxzdTSM5geeBv4DHgM77/cs83fHhzfX2NZcVmbcfNzJymGNyGN8dxOn59Ih1JxTlFNYY7pi9h+zmOtlG0Dct8pjIIh1okM9KX5w9RH6GIYw+eQAbzu+EhkZmHz/0ourEu2Y/KGRYIpBDaBTGpytTnSra+t0iESp05dFxUy4VUWxdIGkF/dItD5lFMHVB8tb8+D8ru8QRyeMfhf8yzFfgvoNWQX7CeZEuoCEpmcoiP9WAuwYzkoJYGiIWFJm1THRqzcqegRO16IXgrbrPn2OncoR1N5mUO/+DwkzmCyWm0EQn8exgWJuYK/AS2GCZW+SFG65brRPq8Z/j/u20D1039gLpCa3IWcOMsYH+WaD58HfDO7AZSaE4jMDIinXYA6dwa2PXJwKY9c2i0RhkRbQPdGKjyyT56tMaRS3AhxtQ2bTUgkPYBbMG0F8ukFB4KCswhAeyuWukky37iqzKRztWqyiuDQhElQGQ2YxMR1twil9F6JuuPoB6InlQpdAzzaQJVYZ54XbOkWPzxj3eHPJ1XsYE6/wC+bqoUDA6qGl0eErGrpT7QeTQrhcGCJjZy+5M5b1R6qpQWNuBhw0Dy9E7hnRz5VGbaZSo0QesdkygirNEcqChynQWea0DB8Np/E31HAeQap5YqshUs9trDjwo03OsIfL6ZOfbo7iFrUHzTSW39ArQIuFmsZSracrJA44ixW+xFikKK8qlWyrXS4LhgPgcjQR8x6AJzOsjoIBLsgGpqbkqZMfBG2gmZcgUbiGwaHeZCaIY/B3JNNl+rDpx2zUA5A53kIy06TRaR1QS2PkStDZXdxjD4pCsoTWEIZgoWx6RZCDTcPHO8usaoDlyncEGdDYRVFXtNL+LaoW7RFsRqUVLbgb5Y+5eUWrHR9YcOJLVnp/q80/5Om8aBbJA6l1ZBp4d75C2wrQHo8+g2njrgzkaZUfswgP/jWeyub+r3zKceCnXsoPd8dtpsIL9A5TVw+ttOgEB3zzLftAwnqm3TUBUKGwqhY2rpy7nINipbwWQnkkSWzzDUqXifEOk91lW71/YVwCpuedRa3H+AhEf1cOeA/6w2236hS0AbJbD1eFyDnw/N/QamG3Evo96zp6TgxzUbXtttBJ9ATTQ6Q71bQL+wpk53SJ6BgLqQQV41kQ5GEd1K4NIIfsJ0EyvMklCrQrFMjiUflCln+YZm/gjvHiK6NLHgrIetdF4VJurcOYWAV1RJiT7kJQDCbuzdYQztsbEeQ5tsDFi84VPIXmBKUPABCXw6bbKytufhwRKqHyBuXxIwCBoEz4IKIcun7nYDk+i+D3rQg90hRh/9NzdHwmCkzKL9cbRIpMjwNiPJV3SrgeYNYGiIrtFn93tcwrQ+WiaCDpzht7UcKO6iL7A92q1rRKT1o5sW7Y9s3oOcpB54N1QiTMR0EtGw2RnrvOAWJG+eggOyuMJKEltclUFYHYKJs1T8SkFqkIi5TNiqwpTVloMy6WBCUZ1DsuTyAZY78PMl2+PuMR6dHnsAu3cGjQLsPpQQeuiTfr+OOrd0X1Lx2Z6dzv7lp/f30Dvkseve6wh9+7E3n08JZwWrMkoor0pdZcgzRFKhoRRiiGmEorUS1ZhRIUZVBOMRc0+lQUakHqA9RzSPbuPONSNIsk1tdOwMsY1LQa0HFX2eYNXp6oWtynuZ87dpw4/+kgqizuWmZeTMBedZsF9BBINaASx1kOOlg5ZpDpyF7K65+WYi2WEPAGI27P3Huz/f3b/90RYZsrHq8Cxs1b06vYzw7zxvZ/jrs9eCaW5/ezImoiX7XOVoElBhJQqbqRTqPWPvRErHpCX3jOAdvfVlRPIbukDUoNijjwydqwTcdih8WAdcLCEi8aGVOpdbZQvAIVKLlSYVQ3yI5FJw8Y/PIsdgSky+uXHl7pM6/1QmDykCPkOexcqHWK9VUC8bEEWGrzw00Cd35B4plMSxjEeQt8o8JV/SVSKh19Oo1Ea//NgrFYGNvgoxcD9DhfUJ7nyjQ9WIubtzvKV+JA6M5TuMe4jJpoDxzl6qe3DUh+5qehXddGmy8C+d8w0pp5wymktp9jUBZN9XnExhZneh2yQ8oAmYVLuJkvmsMYJmm532NlqJzj5XspLNPlKNw6vKNbM7CQmY4AxEFi7MlvthDjR6fIf3EDt8ZRBx7uMro2Ur410EW8N3EBH/rqm7XgZLJZMYOkhMk1zZdzEk6ZOIoQHd2Vcga+gjKGG6OWZID3VmyYMmMBW6jkBz6BczfMyHaIf0+gN6aojlnVUrqIAkXcjYPQEoXjJuqJhGThPtixFCGHDykWfPYKp9coRy3OyU42bgHVo0x/Fyxx/c0KFtpz6hmyEoUOD8bg1Mm0Is5Ky9hD0D78TywFY09noYLxCvQKyMB1+ITqeY1C9Ejn7AeJKDwkm2ZgYhSs/Qs9GudIeJL8G2oaD/ViroFhgW+syy7rTUFCd+JwBRjWgdCpatE5i1eP3d7/nQvZ4M7dyjctVVnvZd3LwuSaDhWj7ECui09/d4n0iZDODz7+FP6q27zOi9aXUZlRJ0JxZRrk4wKqdQIiQBPo7l9hXgKplt5DEPmQLqP9bzSPdKo+N67COGewrxTbQMWNPjBUzMIQdXpRzQywY87RopwAm9F1Rk0L5LbYDekNsatRsInJ+Qg/Pe28VO4nWbAgVxNSuj174/gm40+409FxDBZjN059kMvGA2QzZmM25DQBE1L5vDt3pV4TXGB5yhSxd46zkTbtnjgwFpFWxVfq6UlrHrJp7uA8544HqHCFvmLfoCiEZwKH2jIiSMeASUPrKNjQgJeiBCbGQu/gtQSwMEFAAAAAgAT7tsXcgfy+pXBQAApgsAAAgAAABzY29yZS5weY1W246cRhB95ysqLUUDCcPuWHFso4wlS/FLHK0tOS8WQagHehi80JDuZndGq/n3VFXDXNaxFJ76UrdTdaoaIcTnsjcKnGlkrWAwqmpK1/Tagqxlo62DVm5U26oKKulkEgSA33Bwu16DJd1kOMByWfetl7ip1EPy1fa6xVOyB92hwLPiwra/D4JPpyPYNq1KodcK/vj88Q76zVdVOhiUgbbRip0+iaYSKYjqYXmL30rEIPrRDaOj09/cToGRj+DU3sGhHw10faVaqJVWRjpVvRXHICC4ja6hsWARdOlStP3TyRB0I0LeKPh49+cXkFfBhK2SFereOCOblow87hqn7CBLRfaaWmM6qihhi+8flDkgLIV5YaOddOUO1F6Wrj3E0OiyHckcuMOgIHRmZCu6d7CKYfXizZvb2/lA+K2YbL+DrrGWYVTQG7T1IFtccrRlP2qH5bPwaHoUweSqcyxJIIQImm7ojQNp6kEaq2KgggRb03eojsWeKDCJVWorx9ZRqYLgw/svn2ENmSgxp3VvDlSGwTSYVsdr2VEAxSAbq2jv9rpAYpzWWERclTuptWpp2Updj8g+WmulKlvsxk5qkQdBgK6h7WUVDtLtopR58Ni4HfSD0nwYg9JlT4lci9Ftl69FRNi3XpY+o9xoNGSEMSFjNiRORbDtPbswf7CFZsubhGgxhNHs3cpOhTKGzeR9skZFC2VEBeLlBr3qChmzXsNmUuXchsTHSdeZwzksZBXm8SIoEpy9Ryym9qUakHgsRMX9XSFU9d6Y3sTwzqHsZnR+H30D+K6fGmfak0ME2Vhqa6lLFeJJDFTWCFRrldeYYFNrh9TVhekfbcyjgZeTI9pbBPA0ZNSXeQpDUisXzp3k0ztQbk+qR9asTT8OpHrBq7CV3aaSKTa5xm6+RSowpac1d820Zhrj+lK70S46+pyR15q8nmI/J4aH1FXO60x0ylpknxV5tlzlyOteO6WdyKOTHo+x9VRPBs5Qa488OgtqJIvl7pBtK3L4GcJsK2zblGr9VGcLXi3yI15hJfyF4GB9AbI8Qh1UmZsLtTDmbDHvWffkDscPl+A+9SFO04LKyGy8n7PPO2Yy7bL7POZc4MKXiQWps48n25xyAv1fhhFdSM4TLNKoLPLVe8CkkHxEXUAbMnnODnmiDJEzztS5MvQRFM+NjG7z68sMiYEJXcPq+bknCt89D/a5qOcRi/Ly6v4qEdehTeqeejmmjSxQAnDJgkh6KsTxREFCECMkqi1jSvCp6DBTZ8MaNTysc4bIwVzSa4c3KP+dUqFznzFS5P7R2CfE8mJuoos03fAtwy/4TfLXc2ro+hvs/GGnSF1wSIUsS1Ibu9CHfEGEG2hxNHPl5261KOsXx6uBNLpp2nT4qzGnRg4IY36ZknemHjtsx0+0M2E0iSSywiCmu1D4/w98QIz6Z2yQBOu/8Dn9rjDR5H8LUx5RWPKbuBbW4WAs6LXGw51qhzW9fpre93KH78fS4F+C3LT+R0ZMZgnSkDAkMm4nIIanhZ+1/MzJhIBg4qYdd5OXxYEh+ccpvZhL6Ni/DtXYDTZEe/RnUWHo69XFXPIJD846W/G0YF4u0hevj/C00Iv07UtaME1w84o2TIp54yvIOxFdEZ0oafFPQVU+gnt1WPuBDvsUwj38sAaeiDHso8uHCsGjwrN2P0VIxz48k2GAefqy8usztfHwVbL60R9fcPrq/Jq389WEwXsTf2v811z63zUUGo0sDxBizMBv3gXg+xgeCDAF7sd8NrM8/7bJZzBALb36BQN6SH+d3AcB1rQoCGZR0MgURUGdUBTCG/BtEfwLUEsDBBQAAAAIAE+7bF1xWaV7uwMAAKoHAAANAAAAdGVzdF9hdWRpdC5weYVVbU/kOAz+zq+I9ktabejOK9BBPWnvZaXRSgjNsR9Ww6gKrYfmSJNukgIV4r+fkzJMebtDI5TYrv3Yj+2IutHGkUI3HfvHasUc1M1WSGCtEs6BdQdbo2vScFdJcUVEb3+O117B21K4ndhAw4WxLAiZ1Lxkt1yK8qCQ3Fry1Ysv0KeNdt4Tf/2DW4gXB6SELbHgfjSRBbn1EuIPSZeVonBRwR1ca9Nl1MC2VSVljRHaCIeS8yllvNatcjlCsJCdaQXM3au8xK/2N1H256LiSoHsL5Kr65ZfQ0ZBUaYASptXbc1V9o1LhNYj24r7gIu5eGHAtUbt8o16lIyuoABxC+WCTEaTo8PRGH8kuqja+FL9rVtTwIJAzYW8VJeKfnbxerTpnfta5DWC6fa5b7VB+b1jd1xhiRVZR/RCOy7J8mxFpukxueHEVTyhDC+jUcwi+jXUYEFWNiGT2fzoiHwXqpQdQrVa3gLaBnFvvXTkjtsQaJxMieQ3lUWL8XQU/rzJRQXEGa4sL5zQivRFJsJiCDJLj5M0xS/wkKavPE5nyfTGY5sFd/HG50TuhKt6Vm175cmPfIqZ/xcvghw7BYz761fLZV9YX/igX9MhxXQTChMPClgKi1gLp5842bdQ73RpPd97r/TbzyWpO1IiJR2RohYhs2kyQ+BvwsUvnb1C+B+V8nSdYKnI++GwkNNjdjQZJ/PRO2HZSSB3mCa2tM25KvNfrcbju7m+hcd7bkrekStAhoB06A0MCpJL9dv+Rpak4FJCSVa/Lz2g3RwhGIqdPT8cTw6nI/o/BVmFKSW1sFao6xAC/REJ19jCfZG8890Qe+c4xsNEXYVtW2lZvhwKxZp+HGZpmvYfsWiOPYbniT97ebq7eIXXjCm24Mdwt4HAJ9J8nZCWB/WYvILYDPEprQaMv95aa7rbWHST0ZuuoKc7Rb+KvPjsfDyZ4kQen6SjJ/0HvToYrfnog/58RmaLChfNW0SD5YbRx8OAYddFYV8/LbR46FGoQteNBAeYtssNXLVCun2EMNi71wPXun8RuOn+FAb8QHZRTHzzhSXQZP4Bicr41GQPFAuxoM4c3lNGa7AWN7Gli/UDNVoCqmyHfVmjssByg3JeRB/Zs75F9C+0bqjF5HAr4KJ4YfLwacfNpwV93DyeRs0XxMAFtmRyh1xD7ldO5N/EpGzrxkYm/kxxa8en4XXb2zM86dah4k1z+Rcw6tVfgnFeSOAq8V4l9uYaORNbkueK15DnWUbzvPZmOV08v5BeEMUH/wJQSwMEFAAAAAgAT7tsXRbyr3RvAAAAgwAAABAAAAByZXF1aXJlbWVudHMudHh0Fc1LDsMgDEXROWupLMhPHdTZSmWoIZEaEmEzyO4D06t39LRQlniWg4sgTjA7GM3FUREtuBkGQyHwnwspIzpYwBq/q1D++VtZ+mx6t7jVlPacIgX+btX3PtrGcz2ue21yWF6f0QhFVs5yFlk7bW8PUEsDBBQAAAAIAE+7bF0NO2HioAAAAOgAAAAQAAAAYmVzdF9jb25maWcuanNvblXNsQ6CMBAG4N2nIJ0FS217wdHN0Uc422sglhYp6GB8dwtMLJf7/vy5+x6KgvXRkmeXgt0/FE7LEJUqeaWu5S2kaZzNxI5L8TVjmGRuOvSJ1mjE8MxBrVehH1rMPIuVfsw7rzjnm2mIpk0525iIbIbcZDxh2N02sR8w/979o3eXurgUGSApBZpLtE7DgxSXNTjVGGGEqzVYcg0CKHb4/QFQSwMEFAAAAAgAT7tsXaYBVDWcCAAAMxIAAAkAAABTQ0hFTUEubWR1WNuS28YRfcdXdFEPIRmQ2YuuW7GrNpZdUWyttiTFVX4ShsCQHBPAIDMDUojllzzkA/I3+R1/SU73ACS43qhKIjDomb6dPt2jJ3RnqlXr71VHH51RG00/qJUuS1Nv6EOjc5rur1LKW+d0HWZJchI3nhStTR6MrVVJjeoqiGCxaZb07V47iNS5rfgk3zaNdYGCyXc68NbQuloXkAiWbK3pbx/e3ZFd/azzkFLYapqHaI7TuXXFfEkft9hW2LxlNXIEpDxOL/FjW5drsmsKrg1bWltHh60KsBC7cUToz6HS2p2n0uz0kubz27KkQu9J1QUF7QOV7LzH/rK0B2iAGs9R0J9VHspuOZ8nyZMn9KZu2pAkWZYl73WuzV4XN3R1cfV8cfFycfmUpt85M0s+iFU3pCtlyiT5Mxuctz7YSrs/eKq09/DwazkmyYaDssG3QgUtD33UlHP8fUlZPFgEOXZwO0PUFzk8zlLKRB8/hIMJQTt+NHu3CE7VPnemCdlS3HjXBvHjdojjKAuEvVuEqPecDfGaXtJOd36OdPwdb1ndlmWGSOsakd6rstVsU20DNQ7inKd65MJSQvazt3XyyySHexvruskNTZxet3UxSWnSOGOdCbJ6f8UrqrJtHT41yniN1atX18+vLy7wIXyuP3GMWDTG/tXi8nrSfzEFr9/dP3v14urFs5cXz5/yF4SornXJn2odVqrewXH+UKp60yIb/EXXvFJrXfhP27ZSNRbXqvT615iqL/S97ugLfewajZ9bxgqwLP57+pJ8WSwWx7+QzgZXM0h74BoVgdW+Yj6tkS2kHUmKYeCntVOtPKg8F/fxC7jwyq7L+ceu19rJgkWAXUaiaQjfuab7S5a7v5J/r3vRcVxZHKWoN9oRSofTipUowSkUIZr+9q//XtJXdHlxEVdm8aQhESOlp0Oyn/Bn8fbt4vXr7CRuiseF7+6zvvgQ0FUHVVSYjQk+bu3T9/jetjHsYK6cRO6UX347qLLUobdgSPZ5lHTNgluzwEOUG0GARVfWllrVLAua0ZInhgVLczm9b0vt+ekJDRnHKT9KWXyRgpFKGQGElcRUM5xG9EBedcyvUrJKOJamPWBAx8pvpT4Lp+BYihgJxxV6ZcKMtiADzewamALnc+Cjg0ghhVmpnU6ZpJ3daw4fE+OmRjbnA8keTA1iAYOTFhoXgB1dWvZIO0cv1wEdDRSz1xFMwTaLtqEx+cGIQlPcmdJBgdd1jpbDb8y4oc13Ke8djHZa5Vs5AtaaxkBJb0ZfMlD/FlQoXSliR+W7G1Ke8y/tQFEUTbknqDpHj4MYWoJ2qXzkjhRbj1eVPos8YOc2kK6sY0YGBCwOSI9nxkBvVaS+I0/HMjsvYBj6DTAMsQ03wvoGHQnUyJbQqn/st6THhfs3dyk7sbGgc65G73NbSPb4+QAvUnr38b6PVewjKTEFMxPbVakr39vD9AEj3hSIIYiCkGGDLq7Y0RvactezcbFDaraIqDRbLgU+cywe3Y5pZFsgBtwViHh6bNQ+pVtVbJVyf7q/hRd7U2hL3//0TW9OT2MSFr/lrKVscWWJPcRuhKKxNR7QuVQACri+PTuHjQ6Dx8rWrdeycsAnaiyYbPB24Ea6rTt0c2AB7R0tmQ8B6ErDAE/R3plZeNoAyFlpabkjChqO2digplghYrRAjP4BsucoiCop+oF8aYryKrvY+xiQDLJZcskTB3PxfE5mjZmjk4FF+iq3fFDKTUJEi3HDwIeeIeInGQyMR9tmoPQE7XEwuPnZRXoBakYmGKgYVY47RoW308wr83mpN/AlbFFaYT6/IU4We0Pv//IGyAbEMegVHtTHOah9y9sRCRcY96U6dLFw4jkAnskFkPE9Fk5yFV2+6l1meN5fyrSlDWfmcX8fUgtOPdb5fI7dOE0y88D9x7x/cPSDakyuo4HXbKBg5QAiXMZ0fhwxwNrosvDJedNMR60vPfU1du/UqQBijFwrLZMaz5JcPmN2WdJ37ODjjT4e1qM4DlEyHHZgGQSgPHCbiGNYb/XYwmQhTaWPUwTbGbFBcV4qU43HTAmnWtkWsJDUo0QYbIB8HANAc4BHPy4soePNpmZijG0iKvM3Q92AiUslbFmayjAfrLX2vC3bZUAdjIF223r2dJpdLq+x/BXxqJFizpstuV3vtoMoZg9JM0s+o/ill47wn8nRcTAVyOHyAdY4YcUH5KvoozXkr4+UjNyPxOk8PBKclOMwHm5Y73vtbbnnLlWCIffxQE/TSQeu0K5QHU+W11jm7r6x/LZTpQy+ynk7mWFVof0GUTm6E/BBEutoXKeV8G1lvBe+BQXKh8r62ALgrDgeXZJUr/Q6NrBHT34tpkI3GBrkZjnn0FIangP4RgE/uEdLBF1gepQrQvZ1NhM0cnU/DNxw+CkhZ3RUWB17JsadOByZcBxfRikyBc44XT3HGkzhRbuMjn88DYxLurOuUqX5p2bo4gKqHeYmHY1mWDQqZ67fdg0UM0uT76qVLYHdrG7o8ur66TN6/uLlq4uMfvv3f1iDrMUl9uqdIF46kQbEMYlWK3QzmvZY6d/RyDGI/l4ujZ1heI1RnM8RDya4cx9/h+r/Fw1GxQaZrfvw9UTUA7ynVsCkeAzoKdTbujzlAofpz01pcsN3wFg63C3Ohm2a5k4XkOa3NE6h/fNEGm2vFTD/0XiV0lvF1cCfZueDOjpn6S1NmFwc1mm4oc1OI/woEg8Ze5jrWaKf4x8g7nTtZtcOzshAhUb9V74AG4zVUzwVhpd+QAnXFC/MGEosKm2FIFTmMypDUPRtv4cHMB+Jh3U+NGt0jUjizYHNGt3VuDPJHQ3Rd31zO78JCOVv1Z45Hx276JiYA5IGU4b/XBFaihMHWAF3jr7epzz+4hpfNbjhT/AdGQmm0nK5ZraRJiNPvLTCrEb8zwTc+25wZbjqLJP/AVBLAQIUAxQAAAAIAE+7bF0d67jhhRQAAE82AAALAAAAAAAAAAAAAACAAQAAAABwaXBlbGluZS5weVBLAQIUAxQAAAAIAE+7bF2PkqcECQ0AAE4fAAAIAAAAAAAAAAAAAACAAa4UAABhdWRpdC5weVBLAQIUAxQAAAAIAE+7bF3IH8vqVwUAAKYLAAAIAAAAAAAAAAAAAACAAd0hAABzY29yZS5weVBLAQIUAxQAAAAIAE+7bF1xWaV7uwMAAKoHAAANAAAAAAAAAAAAAACAAVonAAB0ZXN0X2F1ZGl0LnB5UEsBAhQDFAAAAAgAT7tsXRbyr3RvAAAAgwAAABAAAAAAAAAAAAAAAIABQCsAAHJlcXVpcmVtZW50cy50eHRQSwECFAMUAAAACABPu2xdDTth4qAAAADoAAAAEAAAAAAAAAAAAAAAgAHdKwAAYmVzdF9jb25maWcuanNvblBLAQIUAxQAAAAIAE+7bF2mAVQ1nAgAADMSAAAJAAAAAAAAAAAAAACAAassAABTQ0hFTUEubWRQSwUGAAAAAAcABwCTAQAAbjUAAAAA"
PROJECT.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(SOURCE_ZIP))) as z:
    z.extractall(PROJECT)
print('Upload the original training_assessment.zip (private input, not committed).')
uploaded=files.upload()
for name,data in uploaded.items():
    if not name.endswith('.zip'): continue
    target=ROOT/'input_upload'
    target.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        for member in z.infolist():
            assert (target/member.filename).resolve().is_relative_to(target.resolve())
        z.extractall(target)
    for source in target.rglob('ASSIGNMENT.md'):
        shutil.copytree(source.parent,PACK,dirs_exist_ok=True)
del uploaded
assert (PACK/'data/train.jsonl').exists(),'Original assessment pack not found'
os.chdir(PROJECT)


## Install and test before training
This installs the CUDA-13-capable bitsandbytes pin and tests actual NF4 quantization. Dependencies are imported in fresh child processes, so this notebook kernel does not need a restart.

In [3]:
def run_live(command, log_path):
    log_path=Path(log_path)
    log_path.parent.mkdir(parents=True,exist_ok=True)
    print('Log:',log_path,flush=True)
    with log_path.open('a') as log:
        with subprocess.Popen(command,cwd=PROJECT,stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:
            try:
                for line in proc.stdout:
                    print(line,end='',flush=True)
                    log.write(line);log.flush()
                rc=proc.wait()
            except KeyboardInterrupt:
                proc.terminate();proc.wait()
                raise
    if rc: raise RuntimeError(f'Command failed ({rc}). Read the visible error or {log_path}')

run_live([sys.executable,'-m','pip','uninstall','-y','gradio','gradio-client','diffusers'],ROOT/'logs/setup.log')
run_live([sys.executable,'-m','pip','install','-r',str(PROJECT/'requirements.txt')],ROOT/'logs/setup.log')
smoke="""
import torch, bitsandbytes as bnb
assert torch.cuda.is_available(), 'Enable T4 GPU'
assert 'T4' in torch.cuda.get_device_name(0), 'Assignment requires T4'
x=torch.randn(128,128,device='cuda',dtype=torch.float16)
q,state=bnb.functional.quantize_4bit(x,quant_type='nf4')
y=bnb.functional.dequantize_4bit(q,state)
torch.cuda.synchronize()
assert y.shape==x.shape and torch.isfinite(y).all()
import pipeline
print('GPU TEST PASSED:',torch.cuda.get_device_name(0),'Torch',torch.__version__,'bitsandbytes',bnb.__version__)
"""
run_live([sys.executable,'-u','-c',smoke],ROOT/'logs/preflight.log')
run_live([sys.executable,'-m','unittest','-v','test_audit'],ROOT/'logs/tests.log')


Log: /content/drive/MyDrive/NimbusPayAssessment_Fresh_v2/logs/setup.log
Found existing installation: gradio 6.31.0
Uninstalling gradio-6.31.0:
  Successfully uninstalled gradio-6.31.0
Found existing installation: gradio_client 2.7.2
Uninstalling gradio_client-2.7.2:
  Successfully uninstalled gradio_client-2.7.2
Found existing installation: diffusers 0.42.1
Uninstalling diffusers-0.42.1:
  Successfully uninstalled diffusers-0.42.1
Log: /content/drive/MyDrive/NimbusPayAssessment_Fresh_v2/logs/setup.log
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 92.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 38.2 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 100.6 MB/s  0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
  Attempting uninstall: tokenizers
    Found existing installation: tokeniz

In [4]:
PRIVATE=ROOT/'private_data'
run_live([sys.executable,'-u','audit.py','--train',str(PACK/'data/train.jsonl'),'--out',str(PRIVATE)],ROOT/'logs/audit.log')
print('Inspect proposed corrections:',PRIVATE/'audit_changes.jsonl')
print('Original data is unchanged. No dev/test rows are used by the cleaner.')


Log: /content/drive/MyDrive/NimbusPayAssessment_Fresh_v2/logs/audit.log
{
  "input_rows": 1445,
  "kept_rows": 1360,
  "dropped_rows": 85,
  "fixed_rows": 218,
  "exact_duplicate_extras": 60,
  "whitespace_normalized_duplicate_extras": 77,
  "long_inputs_over_1000_chars": 40,
  "conflicting_input_groups": 0,
  "reasons": {
    "schema priority precedence": 62,
    "legacy priority key/value": 70,
    "legacy date format": 30,
    "invalid/truncated target JSON; no label reconstruction": 25,
    "normalize transaction ID": 20,
    "unique explicit claim amount": 21,
    "remove quoted earlier emails from training input": 48,
    "exact duplicate of tr-00030": 1,
    "exact duplicate of tr-00150": 1,
    "exact duplicate of tr-00137": 1,
    "exact duplicate of tr-00070": 1,
    "exact duplicate of tr-00142": 1,
    "exact duplicate of tr-00217": 1,
    "exact duplicate of tr-00173": 1,
    "exact duplicate of tr-00345": 1,
    "exact duplicate of tr-00175": 1,
    "exact duplicate of tr

## Reproduce the selected configuration
This cell trains only the selected raw-data LoRA model from a fresh base, then scores dev and generates test predictions. The pipeline contains all five experiment definitions; saved exploration metrics are in evidence/. No test labels are used.


In [5]:
MODE = 'reproduce'
REPRO_NAME = 'reproduction_02'  # Use a new name for a fresh timing check.
OUT=ROOT/'runs'/('exploration' if MODE=='explore' else REPRO_NAME)
BEST=PROJECT/'best_config.json'
if MODE=='reproduce': assert BEST.exists(), 'Finish exploration first'
# Refuse a concurrent copy of this pipeline.
for proc in Path('/proc').iterdir():
    if not proc.name.isdigit(): continue
    try:
        args=(proc/'cmdline').read_bytes().split(b'\0')
        if any(a==b'pipeline.py' or a.decode(errors='ignore').endswith('/pipeline.py') for a in args):
            raise RuntimeError(f'Another pipeline is running (PID {proc.name}); do not launch twice.')
    except (FileNotFoundError,PermissionError,ProcessLookupError): pass
command=[sys.executable,'-u',str(PROJECT/'pipeline.py'),'--mode',MODE,
         '--pack',str(PACK),'--private',str(PRIVATE),'--out',str(OUT),
         '--best-config',str(BEST)]
run_live(command,ROOT/'logs'/f'{MODE}_{time.time_ns()}.log')
print('FINISHED:',MODE)
print((OUT/'runtime.json').read_text())


Log: /content/drive/MyDrive/NimbusPayAssessment_Fresh_v2/logs/reproduce_1794521975533045102.log
I0000 00:00:1794521983.278309    1888 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794521985.790658    1888 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
STARTING: final
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
Training checkpoint: fresh run

  6%|▌         | 10/180 [01:10<18:40,  6.59s/it]
               

In [ ]:
print((OUT/'runtime.json').read_text())
print(json.dumps(json.loads((OUT/'submission/error_summary.json').read_text())['all'],indent=2))
print('Outputs:',OUT/'submission')
print('Elapsed notebook minutes:',round((time.perf_counter()-STARTED)/60,2))


## Evidence and limits
The submitted adapter and predictions are the unchanged artifacts from the original successful Colab run. Fresh training may differ across GPU/software environments. Recorded package versions, exact model revision and original notebook outputs are retained in evidence/. Run infer.py to test the saved adapter directly. The completed report is report.pdf (three pages); report.md contains the same text.
